# Controlling Claude's Output Format

Two levers force Claude to emit **exactly** the shape you need, no preamble, no closing chatter:

- **`stop_sequences`** halts generation the moment Claude produces a matching string. The stop token itself is **not** returned in the response, and `stop_reason` flips to `stop_sequence`.
- **Assistant prefill** pre-seeds the start of the assistant turn. Claude continues from your seed instead of starting fresh, so you can plant an opening ` ```json ` fence and force it straight into the payload.

Combine them and Claude returns a clean JSON body you can hand directly to `json.loads`. The worked example asks for an **Azure Event Grid** subscription filter.

## Attribution

Adapted with thanks from **[jaozc/building-with-the-claude-api](https://github.com/jaozc/building-with-the-claude-api/tree/main)**.

Changes for this course: swapped `%pip install` for the **uv** install pattern (uv-managed venvs ship without pip), set the model to **`claude-haiku-4-5`** (the repo default), and rewrote cloud-specific prompts to **Azure**.

### 1. Install dependencies

In [1]:
# uv-managed venvs ship WITHOUT pip, so `%pip install` fails here.
# Use uv instead, pointed at THIS kernel's interpreter so packages land
# where the kernel actually looks. Idempotent: uv no-ops if already satisfied.
import subprocess, sys

subprocess.run(
    ["uv", "pip", "install", "--python", sys.executable, "anthropic", "python-dotenv"],
    check=True,
)
print("Dependencies ready.")

Dependencies ready.


### 2. Load environment variables from .env file

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

### 3. Create an API client

In [3]:
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"  # unversioned alias -> current Haiku 4.5 snapshot

### 4. Helper functions

The `chat` helper now carries a **`stop_sequences`** parameter. When present, it lands in the `client.messages.create()` call, so any turn can request an early halt. `system` and `temperature` stay optional.

In [4]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message.content[0].text

### 5. Prefill an assistant turn and stop at the closing fence

The request stacks both levers. The **user** turn requests an Azure Event Grid subscription filter as JSON. The **assistant** turn is prefilled with ` ```json `, which forces Claude to continue directly inside the code fence instead of writing a lead-in sentence. The `stop_sequences=["```"]` argument then halts generation the instant Claude tries to close the fence, so the response holds the JSON body and nothing else.

> **Gotcha:** the prefill text is part of the assistant turn, but the API response returns only Claude's **continuation**, not your seed. So `text` starts with the JSON body, not with ` ```json `. That is exactly why `json.loads(text.strip())` parses cleanly in the next cell.

In [5]:
messages = []

add_user_message(messages, "Generate a very short Azure Event Grid subscription filter as json")
add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])

### 6. Parse the response into a Python dict

Because the stop sequence trimmed the closing fence and the prefill suppressed any prose, `text` is a raw JSON string. A single `json.loads` turns it into a native Python dict, ready to feed straight into an Azure SDK call or a config file.

In [6]:
import json

clean_json = json.loads(text.strip())

clean_json


{'filter': {'subjectBeginsWith': '/blobServices/default/containers/mycontainer',
  'includedEventTypes': ['Microsoft.Storage.BlobCreated']}}